In [1]:
import duckdb
import pandas as pd
from pathlib import Path

con = duckdb.connect("../instacart.duckdb")

Query 01 - How many unique users are included in the dataset?

In [16]:
query_01 = Path("../sql/query_01.sql").read_text()

print(query_01)

print(con.execute(query_01).df())

SELECT COUNT(DISTINCT user_id) AS users_count
FROM orders;

   users_count
0       206209


Query 02 - How many orders have been placed?

In [17]:
query_02 = Path("../sql/query_02.sql").read_text()

print(query_02)

print(con.execute(query_02).df())

SELECT COUNT(DISTINCT order_id) as order_count
FROM orders;
   order_count
0      3421083


Query 03 - How many products belong to each department?

In [22]:
query_03 = Path("../sql/query_03.sql").read_text()

print(query_03)

print(con.execute(query_03).df())

SELECT d.department, COUNT(p.product_id) AS product_count
FROM products p
LEFT JOIN departments d ON p.department_id = d.department_id
GROUP BY d.department
ORDER BY COUNT(p.product_id) DESC;
         department  product_count
0     personal care           6563
1            snacks           6264
2            pantry           5371
3         beverages           4365
4            frozen           4007
5        dairy eggs           3449
6         household           3085
7      canned goods           2092
8   dry goods pasta           1858
9           produce           1684
10           bakery           1516
11             deli           1322
12          missing           1258
13    international           1139
14        breakfast           1115
15           babies           1081
16          alcohol           1054
17             pets            972
18     meat seafood            907
19            other            548
20             bulk             38


Query 04 - What are the 10 most frequently purchased products?

In [8]:
query_04 = Path("../sql/query_04.sql").read_text()

print(query_04)

print(con.execute(query_04).df())

SELECT p.product_name as name, count(*) as purchase_count
FROM order_products o
LEFT JOIN products p ON o.product_id = p.product_id
GROUP BY p.product_name
ORDER BY purchase_count DESC
LIMIT 10;
                     name  purchase_count
0                  Banana          491291
1  Bag of Organic Bananas          394930
2    Organic Strawberries          275577
3    Organic Baby Spinach          251705
4    Organic Hass Avocado          220877
5         Organic Avocado          184224
6             Large Lemon          160792
7            Strawberries          149445
8                   Limes          146660
9      Organic Whole Milk          142813


Query 05 - What are the 10 least frequently purchased products (with at least one purchase)?

In [25]:
query_05 = Path("../sql/query_05.sql").read_text()

print(query_05)

print(con.execute(query_05).df())

SELECT p.product_name as name, count(*) as purchase_count
FROM order_products o
LEFT JOIN products p ON o.product_id = p.product_id
GROUP BY p.product_id, p.product_name
HAVING purchase_count >= 1
ORDER BY purchase_count ASC
LIMIT 10;
                           name  purchase_count
0         Strawberry Energy Gel               1
1  Pasta Shapes In Tomato Sauce               1
2          Yellow Fish Breading               1
3     Orangemint Flavored Water               1
4                 Brut Prosecco               1
5                Original Lager               1
6             Jamaican Allspice               1
7             Salsa, Black Bean               1
8      Miss Treated Conditioner               1
9          Escapes Variety Pack               1


Query 06 - What is the average basket size?

In [18]:
query_06 = Path("../sql/query_06.sql").read_text()

print(query_06)

print(con.execute(query_06).df())

SELECT AVG(basket_size) as avg_basket_size
FROM(
    SELECT COUNT(product_id) as basket_size
    FROM order_products
    GROUP BY order_id
);
   avg_basket_size
0        10.107073


Query 07 - What is the largest basket in the dataset?

In [19]:
query_07 = Path("../sql/query_07.sql").read_text()

print(query_07)

print(con.execute(query_07).df())

SELECT MAX(basket_size) as max_basket_size
FROM(
    SELECT COUNT(product_id) as basket_size
    FROM order_products
    GROUP BY order_id
);
   max_basket_size
0              145


Query 08 - What is the distribution of basket sizes?

In [24]:
query_08 = Path("../sql/query_08.sql").read_text()

print(query_08)

print(con.execute(query_08).df())

WITH baskets AS (
    SELECT COUNT(product_id) as basket_size
    FROM order_products
    GROUP BY order_id
    )
SELECT basket_size, COUNT(*) as distribution
FROM baskets
GROUP BY basket_size
ORDER BY basket_size ASC;
     basket_size  distribution
0              1        163593
1              2        194361
2              3        215060
3              4        230299
4              5        237225
..           ...           ...
108          116             1
109          121             1
110          127             1
111          137             1
112          145             1

[113 rows x 2 columns]


Query 09 - Which departments have sold the largest number of products?

In [32]:
query_09 = Path("../sql/query_09.sql").read_text()

print(query_09)

print(con.execute(query_09).df())

SELECT p.department_id, d.department, COUNT(p.product_id) as product_count
FROM order_products o
LEFT JOIN products p ON o.product_id = p.product_id
LEFT JOIN departments d ON p.department_id = d.department_id
GROUP BY p.department_id, d.department
ORDER BY product_count DESC
LIMIT 5;
   department_id  department  product_count
0              4     produce        9888378
1             16  dairy eggs        5631067
2             19      snacks        3006412
3              7   beverages        2804175
4              1      frozen        2336858


Query 10 - Which aisles have the highest sales volume?

In [33]:
query_10 = Path("../sql/query_10.sql").read_text()

print(query_10)

print(con.execute(query_10).df())

SELECT 
a.aisle as aisle_name,
COUNT(o.product_id) as sales_volume
FROM order_products o
LEFT JOIN products p on o.product_id = p.product_id
LEFT JOIN aisles a on p.aisle_id = a.aisle_id
GROUP BY a.aisle
ORDER BY sales_volume DESC
LIMIT 5;
                   aisle_name  sales_volume
0                fresh fruits       3792661
1            fresh vegetables       3568630
2  packaged vegetables fruits       1843806
3                      yogurt       1507583
4             packaged cheese       1021462


Query 11 - Which products are most frequently added to the cart first?

In [45]:
query_11 = Path("../sql/query_11.sql").read_text()

print(query_11)

print(con.execute(query_11).df())

SELECT o.product_id, p.product_name, COUNT(*) as product_count
FROM order_products o
LEFT JOIN products p on o.product_id = p.product_id
WHERE o.add_to_cart_order = 1
GROUP BY o.product_id, p.product_name
ORDER BY product_count DESC
LIMIT 3;
   product_id            product_name  product_count
0       24852                  Banana         115521
1       13176  Bag of Organic Bananas          82877
2       27845      Organic Whole Milk          32071


Query 12 - Which products are most frequently added to the cart last?

In [46]:
query_12 = Path("../sql/query_12.sql").read_text()

print(query_12)

print(con.execute(query_12).df())


WITH last_item_in_order AS (
    SELECT 
    order_id, 
    MAX(add_to_cart_order) as last_item_added
    FROM order_products
    GROUP BY order_id
)
SELECT o.product_id as ID, p.product_name as product_name, COUNT(*) as product_count
FROM order_products o
JOIN last_item_in_order l ON o.order_id = l.order_id AND o.add_to_cart_order = l.last_item_added
LEFT JOIN products p ON o.product_id = p.product_id
GROUP BY o.product_id, p.product_name
ORDER BY product_count DESC
LIMIT 3;


      ID            product_name  product_count
0  24852                  Banana          31140
1  13176  Bag of Organic Bananas          30770
2  21137    Organic Strawberries          24359


Query 13 - Who are the top 20 users by number of orders?

In [4]:
query_13 = Path("../sql/query_13.sql").read_text()

print(query_13)

print(con.execute(query_13).df())

SELECT user_id, MAX(order_number) as number_of_orders
FROM orders
GROUP BY user_id
ORDER BY number_of_orders DESC
LIMIT 20;
    user_id  number_of_orders
0     20451               100
1    169188               100
2     20126               100
3    170096               100
4    110673               100
5    109020               100
6     15124               100
7    170746               100
8     19625               100
9    168637               100
10    17997               100
11   167957               100
12    21288               100
13   113834               100
14   107451               100
15   172039               100
16   107152               100
17   111739               100
18    20965               100
19   172084               100


Query 14 - Which department has the highest reorder rate?

In [44]:
query_14 = Path("../sql/query_14.sql").read_text()

print(query_14)

print(con.execute(query_14).df())

SELECT 
p.department_id as dept_ID,
d.department as dept_name,
ROUND(AVG(o.reordered)*100,2) as reorder_rate
FROM order_products o
LEFT JOIN products p ON o.product_id = p.product_id
LEFT JOIN departments d ON p.department_id = d.department_id
GROUP BY p.department_id, d.department
ORDER BY reorder_rate DESC
LIMIT 1;
   dept_ID   dept_name  reorder_rate
0       16  dairy eggs         67.02


Query 15 - Segment users based on the total number of orders they placed.

In [28]:
query_15 = Path("../sql/query_15.sql").read_text()

print(query_15)

print(con.execute(query_15).df())

WITH orders_by_user AS (
    SELECT 
    user_id,
    COUNT(order_id) as order_count
    FROM orders
    GROUP BY user_id
),
segmented_users AS (
    SELECT
    CASE
        WHEN order_count BETWEEN 1 AND 5 THEN '1-5 orders'
        WHEN order_count BETWEEN 6 AND 10 THEN '6-10 orders'
        WHEN order_count BETWEEN 11 AND 20 THEN '11-20 orders'
        ELSE 'More than 20 orders'
    END AS order_segment,
    COUNT(user_id) as users_count,
    CASE
        WHEN order_count BETWEEN 1 AND 5 THEN 1
        WHEN order_count BETWEEN 6 AND 10 THEN 2
        WHEN order_count BETWEEN 11 AND 20 THEN 3
        ELSE 4
    END AS segments_order
    FROM orders_by_user
    GROUP BY order_segment, segments_order
)
SELECT order_segment, users_count
FROM segmented_users
ORDER BY segments_order;
         order_segment  users_count
0           1-5 orders        43576
1          6-10 orders        60937
2         11-20 orders        50965
3  More than 20 orders        50731


Query 16 - What percentage of purchased products are reordered?

In [34]:
query_16 = Path("../sql/query_16.sql").read_text()

print(query_16)

print(con.execute(query_16).df())

SELECT
ROUND(AVG(reordered)*100,2) || '%' as reorder_rate
FROM order_products;
  reorder_rate
0       59.01%


Query 17 - Which aisles have the highest reorder rates?

In [45]:
query_17 = Path("../sql/query_17.sql").read_text()

print(query_17)

print(con.execute(query_17).df())

SELECT
a.aisle as aisle_name,
ROUND(AVG(o.reordered)*100,2) as reorder_rate
FROM order_products o
LEFT JOIN products p ON o.product_id = p.product_id
LEFT JOIN aisles a ON p.aisle_id = a.aisle_id
GROUP BY aisle_name
ORDER BY reorder_rate DESC
LIMIT 3;
                      aisle_name  reorder_rate
0                           milk         78.18
1  water seltzer sparkling water         72.99
2                   fresh fruits         71.88


Query 18 - What is the average basket size for each department?

In [47]:
query_18 = Path("../sql/query_18.sql").read_text()

print(query_18)

print(con.execute(query_18).df())

WITH basket_by_department AS (
    SELECT
    o.order_id as orderID,
    d.department as department_name,
    COUNT(p.product_id) as product_count
    FROM order_products o
    LEFT JOIN products p ON o.product_id = p.product_id
    LEFT JOIN departments d ON p.department_id = d.department_id
    GROUP BY o.order_id, d.department
)
SELECT
department_name,
ROUND(AVG(product_count), 2) as avg_basket_size
FROM basket_by_department
GROUP BY department_name
ORDER BY avg_basket_size DESC;
    department_name  avg_basket_size
0           produce             3.95
1        dairy eggs             2.49
2            babies             2.38
3            snacks             2.08
4            frozen             1.90
5         beverages             1.85
6           alcohol             1.81
7            pantry             1.68
8              pets             1.65
9      canned goods             1.57
10        household             1.57
11  dry goods pasta             1.45
12    personal care            

Query 19 - Find the top 3 best-selling products within each department.

In [42]:
query_19 = Path("../sql/query_19.sql").read_text()

#print(query_19)

print(con.execute(query_19).df())

   department                                      product_name  \
0     alcohol                                   Sauvignon Blanc   
1     alcohol                                Cabernet Sauvignon   
2     alcohol                                        Chardonnay   
3      babies  Baby Food Stage 2 Blueberry Pear & Purple Carrot   
4      babies             Spinach Peas & Pear Stage 2 Baby Food   
..        ...                                               ...   
58    produce                            Bag of Organic Bananas   
59    produce                              Organic Strawberries   
60     snacks              Lightly Salted Baked Snap Pea Crisps   
61     snacks                            Original Veggie Straws   
62     snacks                               Sea Salt Pita Chips   

    product_count  product_rank  
0            8541             1  
1            6352             2  
2            6346             3  
3            9103             1  
4            8303        

Query 20 - Rank all products by total number of purchases.

In [9]:
query_20 = Path("../sql/query_20.sql").read_text()

print(query_20)

print(con.execute(query_20).df())

SELECT p.product_name,
COUNT(o.product_id) AS product_count,
RANK() OVER (ORDER BY product_count DESC) as ranking
FROM order_products o
LEFT JOIN products p ON o.product_id = p.product_id
GROUP BY o.product_id, p.product_name
LIMIT 10;
             product_name  product_count  ranking
0                  Banana         491291        1
1  Bag of Organic Bananas         394930        2
2    Organic Strawberries         275577        3
3    Organic Baby Spinach         251705        4
4    Organic Hass Avocado         220877        5
5         Organic Avocado         184224        6
6             Large Lemon         160792        7
7            Strawberries         149445        8
8                   Limes         146660        9
9      Organic Whole Milk         142813       10


Query 21 - Which users belong to the top 10% by number of orders?

In [14]:
query_21 = Path("../sql/query_21.sql").read_text()

print(query_21)

print(con.execute(query_21).df())

WITH users_ranked AS (
    SELECT user_id,
    COUNT(order_id) as order_count,
    PERCENT_RANK() OVER(ORDER BY order_count DESC) as percent_rank
    FROM orders
    GROUP BY user_id
)
SELECT *
FROM users_ranked
WHERE percent_rank <= 0.10
       user_id  order_count  percent_rank
0         5296          100      0.000000
1       146147          100      0.000000
2       105213          100      0.000000
3       106755          100      0.000000
4        21469          100      0.000000
...        ...          ...           ...
20639    88463           38      0.095457
20640   156628           38      0.095457
20641   177562           38      0.095457
20642    78089           38      0.095457
20643    61797           38      0.095457

[20644 rows x 3 columns]


Query 22 - Assign a sequential order number to each user's purchases using window functions only.

In [44]:
query_22 = Path("../sql/query_22.sql").read_text()

print(query_22)

print(con.execute(query_22).df())

SELECT user_id,
order_id,
ROW_NUMBER() OVER (
    PARTITION BY user_id
    ORDER BY order_number ASC
) as generated_order_number
FROM orders
         user_id  order_id  generated_order_number
0             55   2080943                       1
1             55   2006658                       2
2             55   3057843                       3
3             55   3114269                       4
4             55   2986671                       5
...          ...       ...                     ...
3421078   141017    333162                       7
3421079   141017   2396974                       8
3421080   141017   2098038                       9
3421081   141017   2408901                      10
3421082   141027   1327279                       1

[3421083 rows x 3 columns]


Query 23 - Compare each basket size with the customer's previous basket.

In [34]:
query_23 = Path("../sql/query_23.sql").read_text()

print(query_23)

print(con.execute(query_23).df())

SELECT r.user_id as user_id,
o.order_id as order_id,
r.order_number as order_number,
COUNT(*) as basket_size,
basket_size - LAG(basket_size, 1, 0) OVER (PARTITION BY user_id ORDER BY order_number) AS basket_size_change
FROM order_products o
JOIN orders r ON o.order_id = r.order_id
GROUP BY r.user_id, o.order_id, r.order_number
ORDER BY user_id ASC, order_number ASC
         user_id  order_id  order_number  basket_size  basket_size_change
0              1   2539329             1            5                   5
1              1   2398795             2            6                   1
2              1    473747             3            5                  -1
3              1   2254736             4            5                   0
4              1    431534             5            8                   3
...          ...       ...           ...          ...                 ...
3346078   206209   2266710            10            9                   6
3346079   206209   1854736            11

Query 24 - Which users consistently increase the size of their baskets over time?

In [46]:
query_24 = Path("../sql/query_24.sql").read_text()

print(query_24)

print(con.execute(query_24).df())

WITH users_by_basket_size AS (
    SELECT r.user_id as user_id,
    o.order_id as order_id,
    r.order_number as order_number,
    COUNT(*) as basket_size,
    basket_size - LAG(basket_size, 1, 0) OVER (PARTITION BY user_id ORDER BY order_number) AS basket_size_change
    FROM order_products o
    JOIN orders r ON o.order_id = r.order_id
    GROUP BY r.user_id, o.order_id, r.order_number
    ORDER BY user_id ASC, order_number ASC
)
SELECT 
user_id,
COUNT(order_id) as orders_count,
MIN(basket_size_change) AS min_basket_size_change
FROM users_by_basket_size
GROUP BY user_id
HAVING min_basket_size_change > 0 AND orders_count >= 2;
      user_id  orders_count  min_basket_size_change
0        1117             5                       1
1        1278             5                       1
2        1471             4                       2
3        1560             3                       1
4        1797             3                       1
...       ...           ...                     ...

Query 25 - Find users who place orders regularly with less than 7 days between purchases.

In [41]:
query_25 = Path("../sql/query_25.sql").read_text()

print(query_25)

print(con.execute(query_25).df())

SELECT user_id,
AVG(days_since_prior_order) as avg_days_between_orders,
COUNT(order_id) as orders_count
FROM orders
GROUP BY user_id
HAVING orders_count > 3 AND avg_days_between_orders < 7
       user_id  avg_days_between_orders  orders_count
0        31093                 2.525253           100
1        31118                 3.141414           100
2        31154                 4.369231            66
3        31283                 3.541667            25
4        31396                 4.694915            60
...        ...                      ...           ...
23343   203523                 5.654545            56
23344   203540                 6.333333            10
23345   203585                 5.250000             5
23346   203704                 5.333333             4
23347   204075                 6.381818            56

[23348 rows x 3 columns]


Query 26 - Which products have been purchased by more than 10% of all users?

In [35]:
query_26 = Path("../sql/query_26.sql").read_text()

#print(query_26)

print(con.execute(query_26).df())

    product_id                       product_name  users_rate
0        24852                             Banana        0.37
1        13176             Bag of Organic Bananas        0.32
2        21137               Organic Strawberries        0.30
3        21903               Organic Baby Spinach        0.28
4        47626                        Large Lemon        0.24
5        26209                              Limes        0.23
6        47209               Organic Hass Avocado        0.22
7        16797                       Strawberries        0.22
8        47766                    Organic Avocado        0.21
9        39275                Organic Blueberries        0.19
10       24964                     Organic Garlic        0.18
11       22935               Organic Yellow Onion        0.17
12       45007                   Organic Zucchini        0.16
13       27966                Organic Raspberries        0.16
14        4605                      Yellow Onions        0.15
15      

Query 27 - Which department generates the highest number of reordered products?

In [40]:
query_27 = Path("../sql/query_27.sql").read_text()

print(query_27)

print(con.execute(query_27).df())

WITH reordered_products AS (
    SELECT product_id
    FROM order_products
    WHERE reordered = 1
)
SELECT
d.department as department,
COUNT(rp.product_id) as product_count
FROM reordered_products rp
LEFT JOIN products p ON rp.product_id = p.product_id
LEFT JOIN departments d ON p.department_id = d.department_id
GROUP BY d.department
ORDER BY product_count DESC
LIMIT 1;

  department  product_count
0    produce        6432596


Query 28 - Find the product with the highest reorder rate within each department.

In [50]:
query_28 = Path("../sql/query_28.sql").read_text()

print(query_28)

print(con.execute(query_28).df())

WITH product_reorder_rate AS (
    SELECT
    o.product_id as product_id,
    p.department_id as department_id,
    AVG(o.reordered) as reorder_rate
    FROM order_products o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY o.product_id, p.department_id
),
ranked_products AS (
    SELECT
    department_id,
    product_id,
    reorder_rate,
    RANK() OVER(PARTITION BY department_id ORDER BY reorder_rate DESC) as ranked
    FROM product_reorder_rate
)
SELECT d.department, p.product_name, rp.reorder_rate
FROM ranked_products rp
JOIN departments d ON rp.department_id = d.department_id
JOIN products p ON rp.product_id = p.product_id
WHERE ranked = 1
         department                                       product_name  \
0            bakery                               Wheat Sandwich Bread   
1     personal care           Serenity Ultimate Extrema Overnight Pads   
2      meat seafood                                          Bockwurst   
3         breakfast                

Query 29 - Which pair of products is purchased together most frequently?

In [3]:
query_29 = Path("../sql/query_29.sql").read_text()

print(query_29)

print(con.execute(query_29).df())

SELECT 
t1.product_id as product_1,
t2.product_id as product_2,
COUNT(*) as pair_count
FROM order_products t1, order_products t2
WHERE t1.order_id < 1000000 
AND t1.order_id = t2.order_id 
AND t1.product_id < t2.product_id
GROUP BY product_1, product_2
ORDER BY pair_count DESC
LIMIT 10;
   product_1  product_2  pair_count
0      13176      21137       18968
1      13176      47209       18818
2      21137      24852       17030
3      24852      47766       16109
4      21903      24852       15605
5      13176      21903       15274
6      16797      24852       12657
7      24852      47626       12571
8      21137      47209       12378
9      13176      27966       12301


Query 30 - Which combination of three products is purchased together most frequently?

In [6]:
query_30 = Path("../sql/query_30.sql").read_text()

print(query_30)

print(con.execute(query_30).df())

SELECT 
t1.product_id as product_1,
t2.product_id as product_2,
t3.product_id as product_3,
COUNT(*) as tri_count
FROM order_products t1
JOIN order_products t2 ON t1.order_id = t2.order_id
JOIN order_products t3 ON t1.order_id = t3.order_id
WHERE t1.order_id < 10000 
AND t1.product_id < t2.product_id
AND t2.product_id < t3.product_id
GROUP BY product_1, product_2, product_3
ORDER BY tri_count DESC
LIMIT 10;
   product_1  product_2  product_3  tri_count
0      13176      27966      47209         42
1      13176      21137      47209         36
2      21137      21903      24852         33
3      13176      21903      47209         32
4      13176      21137      21903         32
5      21137      24852      47209         32
6       5876      13176      47209         29
7      13176      21137      27966         27
8      24852      26209      47766         26
9      21903      24852      47766         24


Query 31 - Which products are most commonly purchased before another specific product?

In [13]:
query_31 = Path("../sql/query_31.sql").read_text()

#print(query_31)

print(con.execute(query_31).df())

   before_product_id     before_product_name  next_product_id  \
0              13176  Bag of Organic Bananas            21903   
1              13176  Bag of Organic Bananas            21137   
2              13176  Bag of Organic Bananas            47209   
3              21137    Organic Strawberries            13176   
4              24852                  Banana            28204   
5              24852                  Banana            21137   
6              24852                  Banana            47766   
7              47209    Organic Hass Avocado            13176   
8              47626             Large Lemon            26209   
9              47766         Organic Avocado            24852   

        next_product_name  pair_count  
0    Organic Baby Spinach        5858  
1    Organic Strawberries        8191  
2    Organic Hass Avocado        9940  
3  Bag of Organic Bananas        6492  
4      Organic Fuji Apple        7068  
5    Organic Strawberries        7713  
6   

Query 32 - Which products are most frequently purchased together with bananas?

In [50]:
query_32 = Path("../sql/query_32.sql").read_text()

#print(query_32)

print(con.execute(query_32).df())

   product_id          product_name  product_count
0       21137  Organic Strawberries         134535
1       21903  Organic Baby Spinach         113709
2       47209  Organic Hass Avocado         106051
3       47766       Organic Avocado          85254
4       27966   Organic Raspberries          72327


Query 33 - Which user is the most loyal to a single department?

In [62]:
query_33 = Path("../sql/query_33.sql").read_text()

print(query_33)

print(con.execute(query_33).df())

WITH table1 AS (
    SELECT
    o.user_id as user_id,
    p.department_id as department_id,
    COUNT(*) as purchases_in_dept,
    SUM(purchases_in_dept) OVER (PARTITION BY o.user_id) as total_purchases,
    purchases_in_dept/total_purchases as loyalty_rate
    FROM order_products op
    JOIN orders o ON op.order_id = o.order_id
    JOIN products p ON op.product_id = p.product_id
    GROUP BY o.user_id, p.department_id
)
SELECT 
user_id, 
department_id,
total_purchases,
loyalty_rate
FROM table1
WHERE total_purchases >= 50
ORDER BY loyalty_rate DESC, total_purchases DESC
LIMIT 1;
   user_id  department_id  total_purchases  loyalty_rate
0   201038              4            530.0           1.0


Query 34 - Which department has the greatest diversity of products purchased?

In [64]:
query_34 = Path("../sql/query_34.sql").read_text()

print(query_34)

print(con.execute(query_34).df())

SELECT
p.department_id,
d.department,
COUNT(DISTINCT op.product_id) as distinct_product_id
FROM order_products op
JOIN products p ON op.product_id = p.product_id
JOIN departments d ON p.department_id = d.department_id
GROUP BY p.department_id, d.department
ORDER BY distinct_product_id DESC
LIMIT 1;
   department_id     department  distinct_product_id
0             11  personal care                 6563


Query 35 - Which product has the highest reorder rate among products purchased at least 1,000 times?

In [68]:
query_35 = Path("../sql/query_35.sql").read_text()

print(query_35)

print(con.execute(query_35).df())

SELECT
op.product_id,
p.product_name,
COUNT(op.product_id) as product_count,
AVG(op.reordered) as reorder_rate
FROM order_products op
JOIN products p ON op.product_id = p.product_id
GROUP BY op.product_id, p.product_name
HAVING COUNT(op.product_id) >= 1000
ORDER BY reorder_rate DESC
LIMIT 1;
   product_id                     product_name  product_count  reorder_rate
0        9292  Half And Half Ultra Pasteurized           2995      0.861436


Query 36 - For each department, calculate: total number of products, total number of purchases, reorder rate, and percentage share of total sales.

In [86]:
query_36 = Path("../sql/query_36.sql").read_text()

#print(query_36)

print(con.execute(query_36).df())

         department  products_num    sales  reorder_rate  sales_prcnt
0           produce          1684  9888378      0.650521     0.292390
1        dairy eggs          3449  5631067      0.670161     0.166505
2            snacks          6264  3006412      0.574464     0.088897
3         beverages          4365  2804175      0.653651     0.082917
4            frozen          4007  2336858      0.542634     0.069099
5            pantry          5371  1956819      0.347400     0.057861
6            bakery          1516  1225181      0.628381     0.036227
7      canned goods          2092  1114857      0.458639     0.032965
8              deli          1322  1095540      0.608130     0.032394
9   dry goods pasta          1858   905340      0.462220     0.026770
10        household          3085   774652      0.403339     0.022906
11     meat seafood           907   739238      0.568625     0.021859
12        breakfast          1115   739069      0.561351     0.021854
13    personal care 

Query 37 - Find the top 5 best-selling products in each department, including ties.

In [ ]:
query_37 = Path("../sql/query_37.sql").read_text()

print(query_37)

print(con.execute(query_37).df())

Query 38 - For each user, determine: first order date, last order date, and number of days between the first and last order.

In [ ]:
query_38 = Path("../sql/query_38.sql").read_text()

print(query_38)

print(con.execute(query_38).df())

Query 39 - Calculate the median basket size.

In [ ]:
query_39 = Path("../sql/query_39.sql").read_text()

print(query_39)

print(con.execute(query_39).df())

Query 40 - Calculate the rolling average basket size for each user over time.

In [ ]:
query_40 = Path("../sql/query_40.sql").read_text()

print(query_40)

print(con.execute(query_40).df())

In [87]:
con.close()